## Part C — Comparison

In [1]:
import pandas as pd
import numpy as np
import time

# Load the results from Part A and Part B
sklearn_results = pd.read_csv("sklearn_results.csv")
manual_results = pd.read_csv("manual_results.csv")

print("Scikit-learn Results:")
display(sklearn_results)

print("\nFrom-Scratch Results:")
display(manual_results)

Scikit-learn Results:


,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,2.003022,0.051057
1,Logistic Regression,NaN,NaN,NaN,0.708333,0.761194,0.874286,0.81383,0.345022,0.046369



From-Scratch Results:


,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,0.004211,0.000249
1,Logistic Regression,NaN,NaN,NaN,0.716667,0.758454,0.897143,0.82199,0.718100,0.003917


In [2]:
# ============================================
# COMPARISON OF SCIKIT-LEARN AND FROM-SCRATCH
# ============================================

comparison = pd.concat(
    [
        sklearn_results.assign(Implementation="Scikit-learn"),
        manual_results.assign(Implementation="From Scratch")
    ],
    ignore_index=True
)

comparison = comparison[
    [
        "Implementation",
        "Model",
        "MAE",
        "RMSE",
        "R2",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Training Time (s)",
        "Prediction Time (s)"
    ]
]

display(comparison)

,Implementation,Model,MAE,RMSE,R2,Accuracy,Precision,Recall,F1,Training Time (s),Prediction Time (s)
0,Scikit-learn,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,2.003022,0.051057
1,Scikit-learn,Logistic Regression,NaN,NaN,NaN,0.708333,0.761194,0.874286,0.81383,0.345022,0.046369
2,From Scratch,Linear Regression,0.107278,0.14374,0.288521,NaN,NaN,NaN,NaN,0.004211,0.000249
3,From Scratch,Logistic Regression,NaN,NaN,NaN,0.716667,0.758454,0.897143,0.82199,0.718100,0.003917


In [3]:
# Save the comparison table
comparison.to_csv("comparison_results.csv", index=False)

print("Comparison saved to comparison_results.csv")

Comparison saved to comparison_results.csv


In [4]:
# ============================================
# PREPARE DATA FOR OPTIMIZED LOGISTIC REGRESSION
# ============================================

import pandas as pd
import numpy as np
import time

# Load dataset
df = pd.read_csv("data/garments_worker_productivity.csv")

# Remove date
df_model = df.drop(columns=["date"]).copy()

# Create classification target
df_model["MeetsTarget"] = (
    df_model["actual_productivity"] >= df_model["targeted_productivity"]
).astype(int)

# Separate features and target
X = df_model.drop(
    columns=["actual_productivity", "MeetsTarget"]
)

y_cls = df_model["MeetsTarget"].values

# Identify column types
numeric_columns = X.select_dtypes(
    include=["int64", "float64", "bool"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

# Handle missing values
for column in numeric_columns:
    X[column] = X[column].fillna(X[column].median())

for column in categorical_columns:
    X[column] = X[column].fillna(X[column].mode()[0])

# Convert categorical columns to numbers
X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=False,
    dtype=float
)

X_array = X_encoded.to_numpy(dtype=float)

# Load the SAME train/test split used earlier
train_indices = np.load("train_indices.npy")
test_indices = np.load("test_indices.npy")

X_train = X_array[train_indices]
X_test = X_array[test_indices]

y_cls_train = y_cls[train_indices]
y_cls_test = y_cls[test_indices]

# Standardization
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

std[std == 0] = 1

X_train_scaled = (X_train - mean) / std
X_test_scaled = (X_test - mean) / std

# Add intercept
X_train_final = np.column_stack(
    [np.ones(X_train_scaled.shape[0]), X_train_scaled]
)

X_test_final = np.column_stack(
    [np.ones(X_test_scaled.shape[0]), X_test_scaled]
)

# Sigmoid function
def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

print("Data prepared successfully.")
print("Training shape:", X_train_final.shape)
print("Testing shape:", X_test_final.shape)

Data prepared successfully.
Training shape: (957, 25)
Testing shape: (240, 25)


C:\Users\George Mathew\AppData\Local\Temp\ipykernel_19572\804970216.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [5]:
# ============================================
# OPTIMIZED FROM-SCRATCH LOGISTIC REGRESSION
# ============================================

start_time = time.perf_counter()

n_features = X_train_final.shape[1]
optimized_weights = np.zeros(n_features)

learning_rate = 0.05
iterations = 10000
n = X_train_final.shape[0]

for i in range(iterations):

    z = X_train_final @ optimized_weights
    probabilities = sigmoid(z)

    gradient = (
        X_train_final.T @ (probabilities - y_cls_train)
    ) / n

    optimized_weights = (
        optimized_weights - learning_rate * gradient
    )

optimized_training_time = time.perf_counter() - start_time

print("Optimized training completed.")
print("Training time:", optimized_training_time, "seconds")

Optimized training completed.
Training time: 1.3318759000394493 seconds


In [6]:
# ============================================
# EVALUATE OPTIMIZED LOGISTIC REGRESSION
# ============================================

start_time = time.perf_counter()

z_test_optimized = X_test_final @ optimized_weights
probabilities_test_optimized = sigmoid(z_test_optimized)

y_cls_pred_optimized = (
    probabilities_test_optimized >= 0.5
).astype(int)

optimized_prediction_time = time.perf_counter() - start_time

# --------------------------------------------
# Calculate metrics
# --------------------------------------------

optimized_accuracy = np.mean(
    y_cls_pred_optimized == y_cls_test
)

true_positive = np.sum(
    (y_cls_pred_optimized == 1) &
    (y_cls_test == 1)
)

false_positive = np.sum(
    (y_cls_pred_optimized == 1) &
    (y_cls_test == 0)
)

false_negative = np.sum(
    (y_cls_pred_optimized == 0) &
    (y_cls_test == 1)
)

if true_positive + false_positive == 0:
    optimized_precision = 0
else:
    optimized_precision = (
        true_positive /
        (true_positive + false_positive)
    )

if true_positive + false_negative == 0:
    optimized_recall = 0
else:
    optimized_recall = (
        true_positive /
        (true_positive + false_negative)
    )

if optimized_precision + optimized_recall == 0:
    optimized_f1 = 0
else:
    optimized_f1 = (
        2 * optimized_precision * optimized_recall
        / (optimized_precision + optimized_recall)
    )

# --------------------------------------------
# Display results
# --------------------------------------------

print("OPTIMIZED LOGISTIC REGRESSION")
print("=============================")
print("Accuracy :", optimized_accuracy)
print("Precision:", optimized_precision)
print("Recall   :", optimized_recall)
print("F1 Score :", optimized_f1)
print("Training Time (s):", optimized_training_time)
print("Prediction Time (s):", optimized_prediction_time)

OPTIMIZED LOGISTIC REGRESSION
Accuracy : 0.7083333333333334
Precision: 0.7638190954773869
Recall   : 0.8685714285714285
F1 Score : 0.8128342245989304
Training Time (s): 1.3318759000394493
Prediction Time (s): 0.0005786999827250838


In [7]:
# ============================================
# FINAL COMPARISON AND OBSERVATIONS
# ============================================

print("FINAL COMPARISON")
print("================")

print("\n1. Linear Regression")
print("--------------------")
print("Scikit-learn:")
print("MAE :", sklearn_results.loc[0, "MAE"])
print("RMSE:", sklearn_results.loc[0, "RMSE"])
print("R2  :", sklearn_results.loc[0, "R2"])
print("Training Time:", sklearn_results.loc[0, "Training Time (s)"])
print("Prediction Time:", sklearn_results.loc[0, "Prediction Time (s)"])

print("\nFrom Scratch:")
print("MAE :", manual_results.loc[0, "MAE"])
print("RMSE:", manual_results.loc[0, "RMSE"])
print("R2  :", manual_results.loc[0, "R2"])
print("Training Time:", manual_results.loc[0, "Training Time (s)"])
print("Prediction Time:", manual_results.loc[0, "Prediction Time (s)"])

print("\n2. Logistic Regression")
print("----------------------")
print("Scikit-learn:")
print("Accuracy :", sklearn_results.loc[1, "Accuracy"])
print("Precision:", sklearn_results.loc[1, "Precision"])
print("Recall   :", sklearn_results.loc[1, "Recall"])
print("F1       :", sklearn_results.loc[1, "F1"])
print("Training Time:", sklearn_results.loc[1, "Training Time (s)"])
print("Prediction Time:", sklearn_results.loc[1, "Prediction Time (s)"])

print("\nFrom Scratch:")
print("Accuracy :", manual_results.loc[1, "Accuracy"])
print("Precision:", manual_results.loc[1, "Precision"])
print("Recall   :", manual_results.loc[1, "Recall"])
print("F1       :", manual_results.loc[1, "F1"])
print("Training Time:", manual_results.loc[1, "Training Time (s)"])
print("Prediction Time:", manual_results.loc[1, "Prediction Time (s)"])

print("\n3. Optimized From-Scratch Logistic Regression")
print("----------------------------------------------")
print("Accuracy :", optimized_accuracy)
print("Precision:", optimized_precision)
print("Recall   :", optimized_recall)
print("F1       :", optimized_f1)
print("Training Time:", optimized_training_time)
print("Prediction Time:", optimized_prediction_time)

FINAL COMPARISON

1. Linear Regression
--------------------
Scikit-learn:
MAE : 0.1072784330291722
RMSE: 0.1437402979105884
R2  : 0.2885214314434412
Training Time: 2.0030215000151657
Prediction Time: 0.0510569000034593

From Scratch:
MAE : 0.107278433029172
RMSE: 0.1437402979105884
R2  : 0.2885214314434412
Training Time: 0.0042109999922104
Prediction Time: 0.0002485000004526

2. Logistic Regression
----------------------
Scikit-learn:
Accuracy : 0.7083333333333334
Precision: 0.7611940298507462
Recall   : 0.8742857142857143
F1       : 0.8138297872340425
Training Time: 0.3450220999948215
Prediction Time: 0.0463690000178758

From Scratch:
Accuracy : 0.7166666666666667
Precision: 0.7584541062801933
Recall   : 0.8971428571428571
F1       : 0.8219895287958117
Training Time: 0.7180995999951847
Prediction Time: 0.0039166999922599

3. Optimized From-Scratch Logistic Regression
----------------------------------------------
Accuracy : 0.7083333333333334
Precision: 0.7638190954773869
Recall   : 0

In [8]:
# ============================================
# OBSERVATIONS
# ============================================

print("""
OBSERVATIONS
============

1. Linear Regression produced the same MAE, RMSE and R2 values
   for the Scikit-learn and from-scratch implementations.

2. The from-scratch Linear Regression implementation had a shorter
   training and prediction time in this experiment.

3. The Logistic Regression implementations produced slightly
   different classification metrics.

4. The from-scratch Logistic Regression required iterative
   gradient descent for parameter optimization.

5. The optimized Logistic Regression used more iterations and a
   different learning rate, but the metrics did not improve in
   this particular run.

6. Scikit-learn provides optimized and ready-to-use implementations,
   while the from-scratch implementation demonstrates the underlying
   mathematical operations using NumPy.
""")


OBSERVATIONS

1. Linear Regression produced the same MAE, RMSE and R2 values
   for the Scikit-learn and from-scratch implementations.

2. The from-scratch Linear Regression implementation had a shorter
   training and prediction time in this experiment.

3. The Logistic Regression implementations produced slightly
   different classification metrics.

4. The from-scratch Logistic Regression required iterative
   gradient descent for parameter optimization.

5. The optimized Logistic Regression used more iterations and a
   different learning rate, but the metrics did not improve in
   this particular run.

6. Scikit-learn provides optimized and ready-to-use implementations,
   while the from-scratch implementation demonstrates the underlying
   mathematical operations using NumPy.

